# Football Momentum Forecasting — Data Sanity Check

Quick exploration of the raw events dataset before building anything. 
Goal is to understand the dataset we are working with: data shape, text quality, 
event types, and anything that might cause problems downstream.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/archive/events.csv")

## Dataset Overview

The dataset contains 941,000 match events across 9,074 professional football matches from Europe's top five leagues (2011–2017). Each row represents a single in-game event consisting of commentary text and structured match metadata. Before modeling, we verify the data quality and understand its characteristics.

In this section we verify:

- dataset structure
- commentary quality
- missing values
- overall data integrity


The following exploratory checks ensure that the dataset is complete and suitable for downstream feature engineering and sequence modeling.

In [3]:
df.head()

,id_odsp,id_event,sort_order,time,text,event_type,event_type2,side,event_team,opponent,...,player_in,player_out,shot_place,shot_outcome,is_goal,location,bodypart,assist_method,situation,fast_break
0,UFot0hit/,UFot0hit1,1,2,Attempt missed. Mladen Petric (Hamburg) left f...,1,12.0,2,Hamburg SV,Borussia Dortmund,...,NaN,NaN,6.0,2.0,0,9.0,2.0,1,1.0,0
1,UFot0hit/,UFot0hit2,2,4,"Corner, Borussia Dortmund. Conceded by Dennis...",2,NaN,1,Borussia Dortmund,Hamburg SV,...,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,0
2,UFot0hit/,UFot0hit3,3,4,"Corner, Borussia Dortmund. Conceded by Heiko ...",2,NaN,1,Borussia Dortmund,Hamburg SV,...,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,0
3,UFot0hit/,UFot0hit4,4,7,Foul by Sven Bender (Borussia Dortmund).,3,NaN,1,Borussia Dortmund,Hamburg SV,...,NaN,NaN,NaN,NaN,0,NaN,NaN,0,NaN,0
4,UFot0hit/,UFot0hit5,5,7,Gokhan Tore (Hamburg) wins a free kick in the ...,8,NaN,2,Hamburg SV,Borussia Dortmund,...,NaN,NaN,NaN,NaN,0,2.0,NaN,0,NaN,0


In [2]:
df.describe()

,sort_order,time,event_type,event_type2,side,shot_place,shot_outcome,is_goal,location,bodypart,assist_method,situation,fast_break
count,941009.000000,941009.000000,941009.000000,214293.000000,941009.000000,227459.000000,228498.000000,941009.000000,467067.000000,229185.000000,941009.000000,229137.000000,941009.000000
mean,53.858826,49.663663,4.326575,12.233764,1.481170,5.733693,1.926555,0.025978,6.209073,1.624831,0.264332,1.281316,0.004876
std,32.014268,26.488977,2.995313,0.468850,0.499646,3.326100,0.797055,0.159071,5.421736,0.740400,0.655501,0.709394,0.069655
min,1.000000,0.000000,1.000000,12.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,0.000000
25%,27.000000,27.000000,2.000000,12.000000,1.000000,2.000000,1.000000,0.000000,2.000000,1.000000,0.000000,1.000000,0.000000
50%,53.000000,51.000000,3.000000,12.000000,1.000000,5.000000,2.000000,0.000000,3.000000,1.000000,0.000000,1.000000,0.000000
75%,79.000000,73.000000,8.000000,12.000000,2.000000,9.000000,3.000000,0.000000,11.000000,2.000000,0.000000,1.000000,0.000000
max,180.000000,100.000000,11.000000,15.000000,2.000000,13.000000,4.000000,1.000000,19.000000,3.000000,4.000000,4.000000,1.000000


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 941009 entries, 0 to 941008
Data columns (total 22 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   id_odsp        941009 non-null  str    
 1   id_event       941009 non-null  str    
 2   sort_order     941009 non-null  int64  
 3   time           941009 non-null  int64  
 4   text           941009 non-null  str    
 5   event_type     941009 non-null  int64  
 6   event_type2    214293 non-null  float64
 7   side           941009 non-null  int64  
 8   event_team     941009 non-null  str    
 9   opponent       941009 non-null  str    
 10  player         880009 non-null  str    
 11  player2        291310 non-null  str    
 12  player_in      51715 non-null   str    
 13  player_out     51738 non-null   str    
 14  shot_place     227459 non-null  float64
 15  shot_outcome   228498 non-null  float64
 16  is_goal        941009 non-null  int64  
 17  location       467067 non-null  float64


### Inspecting the commentary

The commentary text is the primary input to the NLP models. Before training, we inspect whether it contains meaningful contextual information rather than only short event labels (e.g., "Corner" or "Foul"). Rich descriptions provide stronger signals for learning match momentum.

In [10]:
for i, text in enumerate(df['text'].dropna().head(20)):
    print(f"{i}: {text}\n")

0: Attempt missed. Mladen Petric (Hamburg) left footed shot from the left side of the box is high and wide to the left. Assisted by Gokhan Tore.

1: Corner,  Borussia Dortmund. Conceded by Dennis Diekmeier.

2: Corner,  Borussia Dortmund. Conceded by Heiko Westermann.

3: Foul by Sven Bender (Borussia Dortmund).

4: Gokhan Tore (Hamburg) wins a free kick in the defensive half.

5: Hand ball by Jose Paolo Guerrero (Hamburg).

6: Corner,  Hamburg. Conceded by Lukasz Piszczek.

7: Chris Lowe (Borussia Dortmund) wins a free kick in the defensive half.

8: Foul by Gojko Kacar (Hamburg).

9: Foul by Gokhan Tore (Hamburg).

10: Sven Bender (Borussia Dortmund) wins a free kick on the left wing.

11: Attempt missed. Shinji Kagawa (Borussia Dortmund) right footed shot from outside the box is close, but misses the top right corner. Assisted by Mario Gotze.

12: Foul by Gojko Kacar (Hamburg).

13: Goal!  Borussia Dortmund 1, Hamburg 0. Kevin Grosskreutz (Borussia Dortmund) left footed shot from th

### Inspecting Missing Values

After confirming the commentary quality, we inspect missing values in the columns that will be used for preprocessing and model training. Missing values can lead to incorrect feature generation or failed training pipelines.

In [22]:
df['text'].isna().sum()

np.int64(0)

In [8]:
print(f"Total rows: {len(df)}")

Total rows: 941009


In [9]:
print(f"Rows with text: {df['text'].notna().sum()}")

Rows with text: 941009


In [4]:
df['side'].isna().sum()

np.int64(0)

### Let's look at event type  

We inspect the distribution of event types to verify that the dataset reflects realistic football match events and to understand which events dominate the commentary.

The distribution also provides an intuition about which match events occur most frequently and whether rare events (e.g., red cards) may contribute less during training.

- The majority of events are free kicks and fouls, while second-yellow red cards are comparatively rare.

In [21]:
df['event_type'].value_counts()

event_type
8     237932
3     232925
1     229135
2      91204
7      51738
9      43476
4      39911
10     10730
11      2706
6       1152
5        100
Name: count, dtype: int64

### Inspecting Unique matches

We inspect the number of unique matches to confirm that the dataset covers a diverse set of games rather than repeated commentary from only a few matches.

We also inspect average events per match to check whether the matches have sufficient event density.

- Only average 103 events in the dataset, with standard deviation of 15

In [20]:
# How many unique matches
df['id_odsp'].nunique()

9074

In [18]:
# Average events per match
df.groupby('id_odsp').size().describe()

count    9074.000000
mean      103.703879
std        15.325749
min        32.000000
25%        93.000000
50%       103.000000
75%       114.000000
max       180.000000
dtype: float64

### Time Distribution

We verify that match time is represented in minutes, which is important because momentum features will later depend on temporal ordering.

In [19]:
# Time Distribution
df['time'].describe()

count    941009.000000
mean         49.663663
std          26.488977
min           0.000000
25%          27.000000
50%          51.000000
75%          73.000000
max         100.000000
Name: time, dtype: float64

### Looking at a Sample Match

Inspecting a complete match helps verify that commentary forms a coherent narrative rather than isolated events. This also provides intuition about how momentum evolves throughout a game.

In [15]:
# Sample of one full match 
match = df[df['id_odsp'] == df['id_odsp'].iloc[0]]
print(match[['time', 'text', 'event_type']].to_string())

     time                                                                                                                                                                              text  event_type
0       2                                     Attempt missed. Mladen Petric (Hamburg) left footed shot from the left side of the box is high and wide to the left. Assisted by Gokhan Tore.           1
1       4                                                                                                                         Corner,  Borussia Dortmund. Conceded by Dennis Diekmeier.           2
2       4                                                                                                                         Corner,  Borussia Dortmund. Conceded by Heiko Westermann.           2
3       7                                                                                                                                          Foul by Sven Bender (Borussia Dortmund).           3


### Match Info — Teams and Sides

- Side 1 → Home team
- Side 2 → Away team 


Every event is tagged with which team performed it, 
which is how we'll attribute momentum later.

Before feature engineering, we inspect the match metadata to understand how teams and match sides are represented. This ensures that later calculations correctly distinguish between home and away events.

In [16]:
print(df.columns.tolist())
print(df[['event_team', 'side']].head(20))

['id_odsp', 'id_event', 'sort_order', 'time', 'text', 'event_type', 'event_type2', 'side', 'event_team', 'opponent', 'player', 'player2', 'player_in', 'player_out', 'shot_place', 'shot_outcome', 'is_goal', 'location', 'bodypart', 'assist_method', 'situation', 'fast_break']
           event_team  side
0          Hamburg SV     2
1   Borussia Dortmund     1
2   Borussia Dortmund     1
3   Borussia Dortmund     1
4          Hamburg SV     2
5          Hamburg SV     2
6          Hamburg SV     2
7   Borussia Dortmund     1
8          Hamburg SV     2
9          Hamburg SV     2
10  Borussia Dortmund     1
11  Borussia Dortmund     1
12         Hamburg SV     2
13  Borussia Dortmund     1
14  Borussia Dortmund     1
15  Borussia Dortmund     1
16         Hamburg SV     2
17         Hamburg SV     2
18  Borussia Dortmund     1
19  Borussia Dortmund     1


### Goals Ratio

Goals account for only a small fraction of all recorded events (~3%), reflecting the naturally sparse occurrence of scoring opportunities in football. This suggests that relying solely on goals would be insufficient for modeling momentum.

In [30]:
# Check goals columne
df['is_goal'].value_counts()

is_goal
0    916563
1     24446
Name: count, dtype: int64

### Shot Outcomes

Most shots do not result in goals, which aligns with real match statistics. Shot outcomes may therefore provide useful momentum signals even when no goal is scored.

In [ ]:
df['shot_outcome'].value_counts()

# Off target - most, followed by on-target -> blocked -> hit the post

shot_outcome
2.0    92827
1.0    78014
3.0    54082
4.0     3575
Name: count, dtype: int64

## Summary

The exploratory analysis confirms that the dataset is suitable for building a football momentum prediction model.

Key observations:

- The dataset contains over 941k events across 9,074 matches.
- Commentary consists of rich natural-language descriptions appropriate for NLP.
- Missing values are minimal in the features required for modeling.
- Event frequencies resemble real football matches, with goals being relatively rare.
- Match timelines are complete and contain sufficient event density for sequence modeling.

With the data quality validated, the next step is feature engineering and constructing momentum labels for model training.